# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Observations of Processed Robintrack Data (2018-2020)

---

In [2]:
# load libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# load data
df = pd.read_csv("../../data/processed/robintrack/robintrack_merged.csv")

Once the data are loaded into memory, the goal is to reproduce the summary statistics of Barber et al. (2022).

In [4]:
# convert date to datetime format
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

In [6]:
# head
df.tail()

,date,users_close,users_open,users_11_am,users_2_pm,users_last,intraday_userchg,ticker
6028952,2020-06-11,1289.0,1289.0,1287.0,1288.0,1289.0,0.0,KEM
6028953,2020-06-12,1285.0,1294.0,1291.0,1285.0,1285.0,-9.0,KEM
6028954,2020-06-13,1285.0,1285.0,1285.0,1285.0,1285.0,0.0,KEM
6028955,2020-06-14,1285.0,1285.0,1285.0,1285.0,1285.0,0.0,KEM
6028956,2020-06-15,1285.0,1285.0,1285.0,1285.0,1285.0,0.0,KEM


Once we have pre-processed our data, we drop the NA values (the Robinhood app was down during three distinct periods over the sample).

In [7]:
# drop all rows with missing values
df = df.dropna()
df = df.sort_values(["ticker", "date"])

# summary statistics of users_close and users_last
df["userchg"] = df.groupby("ticker")["users_close"].diff()
df[["users_close", "users_open", "users_11_am", "users_2_pm", "userchg", "intraday_userchg"]].describe().T

,count,mean,std,min,25%,50%,75%,max
users_close,5821130.0,2049.485740,15325.326245,0.0,35.0,160.0,670.0,990622.0
users_open,5821130.0,2045.674381,15304.610355,0.0,35.0,159.0,668.0,990622.0
users_11_am,5821130.0,2046.573076,15311.281689,0.0,35.0,159.0,669.0,990622.0
users_2_pm,5821130.0,2048.159634,15322.564654,0.0,35.0,159.0,669.0,990622.0
userchg,5812537.0,6.490341,306.676996,-548944.0,0.0,0.0,1.0,85193.0
intraday_userchg,5821130.0,3.811359,139.381236,-14584.0,0.0,0.0,0.0,67497.0


In [8]:
# number of unique tickers
df["ticker"].nunique()

8593

In [9]:
# compare unique tickers here with the unique tickers in main data of Barber et al. (2022)
with open("../../data/replication/unique_tickers_main_pseudo.txt", "r") as f:
    unique_tickers_main = set(line.strip() for line in f)

unique_tickers_rh = set(df["ticker"].unique())
overlap_tickers = unique_tickers_main.intersection(unique_tickers_rh)
print(f"Number of unique tickers in Barber et al. (2022) main data: {len(unique_tickers_main)}")
print(f"Number of unique tickers in Robintrack data: {len(unique_tickers_rh)}")
print(f"Number of overlapping unique tickers: {len(overlap_tickers)}")

Number of unique tickers in Barber et al. (2022) main data: 8447
Number of unique tickers in Robintrack data: 8593
Number of overlapping unique tickers: 8443


In [11]:
# users_close = 990622.0
df[df['users_close'] == 990622.0]

,date,users_close,users_open,users_11_am,users_2_pm,users_last,intraday_userchg,ticker,userchg
3555539,2020-05-09,990622.0,990622.0,990622.0,990622.0,990622.0,0.0,ACB,178.0


In [12]:
# filter with overlap tickers
df_overlap = df.copy()
df_overlap = df_overlap[df_overlap["ticker"].isin(overlap_tickers)]

# summary statistics of users_close and users_last for the overlapping tickers
df_overlap[["users_close", "users_last"]].describe().T



,count,mean,std,min,25%,50%,75%,max
users_close,5811198.0,2050.956958,15337.402082,0.0,35.0,160.0,670.0,990622.0
users_last,5811198.0,2051.865106,15344.591348,0.0,35.0,160.0,670.0,990622.0


In [15]:
# write to a .txt file unique_tickers_rh
# one ticker per line
with open("../../data/interim/tickers/unique_tickers_RH.txt", "w") as f:
    for ticker in unique_tickers_rh:
        f.write(f"{ticker}\n")

In [ ]:
# read the .txt file unique_tickers_rh and store it in a list
with open("../../data/interim/tickers/unique_tickers_RH.txt", "r") as f:
    unique_tickers_rh_from_file = [line.strip() for line in f]

print(unique_tickers_rh_from_file[:10])

['SCPH', 'UFCS', 'ESEA', 'TCHP', 'SBRCY', 'IQIN', 'NTST', 'IMMR', 'MYO', 'LEMB']
